# Token Classification:

## Load Dataset:

In [1]:
from datasets import load_dataset

ckpt = "BramVanroy/conll2003"

raw_datasets = load_dataset(ckpt)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [2]:
label_list = raw_datasets['train'].features['ner_tags'].feature.names
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [3]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

## Load Tokenizer:

In [4]:
from transformers import AutoTokenizer

ckpt = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Align Labels:

In [5]:
def tokenize_and_align_labels(examples):
    """Tokenize examples and align NER tags to sub-word tokens.

    - First token of a word  → use the original label (e.g. B-PER)
    - Continuation sub-words → use the I- version of that label (e.g. I-PER)
    - Special tokens         → -100 (ignored by the loss)
    """

    # tokenize inputs:
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            # Special tokens have a word id that is None → ignore.
            if word_id is None:
                label_ids.append(-100)
            # First token of a word → use the original tag.
            elif word_id != previous_word_id:
                label_ids.append(label[word_id])
            # Continuation sub-word → convert B- to I- (same entity, inner token).
            else:
                tag = label_list[label[word_id]]
                if tag.startswith("B-"):
                    label_ids.append(label2id["I-" + tag[2:]])
                else:
                    label_ids.append(label[word_id])
            previous_word_id = word_id
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [6]:
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

## Data Collator:

In [7]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True, 
    pad_to_multiple_of=8, 
    label_pad_token_id=-100
)

## Metrics:

In [8]:
!uv pip install -qq seqeval

In [9]:
from evaluate import load
import numpy as np

seqeval_metric = load("seqeval")

In [10]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert to tag names
    true_predictions = [
        [label_list[pred] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lab] for _, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(
        predictions=true_predictions, references=true_labels
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Load Model:

In [11]:
from transformers import AutoModelForTokenClassification

ckpt = "bert-base-cased"

model = AutoModelForTokenClassification.from_pretrained(
    ckpt, 
    id2label=id2label, 
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

## Finetune model:

In [12]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="bert-finetuned-ner",
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="tankgauravgt/bert-finetuned-ner",
    auto_find_batch_size=True
)

In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.077571,0.859526,0.910300,0.884185,0.977218
2,No log,0.064695,0.903457,0.932346,0.917674,0.982310
3,0.165188,0.060521,0.912232,0.937563,0.924724,0.983899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=660, training_loss=0.1355805982242931, metrics={'train_runtime': 187.1904, 'train_samples_per_second': 225.028, 'train_steps_per_second': 3.526, 'total_flos': 1348270434964512.0, 'train_loss': 0.1355805982242931, 'epoch': 3.0})

In [16]:
results = trainer.evaluate
results()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.165188,0.060521,3,0.912232,0.937563,0.924724,0.983899


{'eval_loss': 0.06052146852016449,
 'eval_precision': 0.9122318650728672,
 'eval_recall': 0.9375631100639515,
 'eval_f1': 0.9247240434890863,
 'eval_accuracy': 0.9838994525225172}